# Agentic AI LLM Project Demo
**Title:** Agentic Tool-Augmented LLMs for Indic Voice-Based Task
Automation

Names:Vaishnav Nigade(2022BCD0045)
Gograj Dhadarwal(2022BEC0057)
Naveen Meghavath(2022BEC0041)

This Colab notebook is a ready-to-run prototype

## What it shows
- Tool-based agentic workflow
- Voice/Text input
- Speech-to-text using Whisper
- Intent detection + tool selection
- Tool execution and final response

## Demo flow
Voice/Text -> Intent -> Tool Selection -> Tool Execution -> Output

## 1. Install dependencies


In [1]:
!pip -q install openai-whisper
!apt-get -qq update
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 8.1 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Define tools
These are the tools your agent can use.

In [16]:
import re
from datetime import datetime

import pytz

def get_time():
    ist = pytz.timezone("Asia/Kolkata")
    current_time = datetime.now(ist)
    return current_time.strftime("%Y-%m-%d %I:%M:%S %p IST")



def calculator(expr):
    expr = expr.lower().strip()

    # convert natural words to symbols
    replacements = {
        "plus": "+",
        "add": "+",
        "minus": "-",
        "subtract": "-",
        "into": "*",
        "multiply": "*",
        "x": "*",
        "times": "*",
        "divide": "/",
        "by": "/"
    }

    for word, symbol in replacements.items():
        expr = expr.replace(word, symbol)

    # keep only safe math chars
    expr = re.sub(r"[^0-9+\-*/(). ]", "", expr)

    try:
        return str(eval(expr))
    except Exception as e:
        return f"Error in calculation: {e}"

def greeting():
    return "Namaste! I am your agentic AI assistant."

TOOLS = {
    "get_time": get_time,
    "calculator": calculator,
    "greeting": greeting,
}

## 3. Agent logic
This is the core of your project.

In [17]:
def detect_intent(user_input: str):
    text = user_input.lower()

    if any(word in text for word in ["time", "date", "samay", "abhi kitna baje", "time kya hai"]):
        return "get_time"

    if any(op in text for op in ["+", "-", "*", "/"]) or any(word in text for word in ["calculate", "sum", "add", "minus", "multiply", "divide", "plus", "guna", "jod", "karo"]):
        return "calculator"

    if any(word in text for word in ["hello", "hi", "namaste"]):
        return "greeting"

    return "unknown"

def run_agent(user_input: str):
    intent = detect_intent(user_input)

    if intent == "get_time":
        result = TOOLS["get_time"]()
    elif intent == "calculator":
        result = TOOLS["calculator"](user_input)
    elif intent == "greeting":
        result = TOOLS["greeting"]()
    else:
        result = "Sorry, I cannot handle this request yet."

    return {
        "input": user_input,
        "intent": intent,
        "output": result
    }

## 4. Quick text demo
 This is live demo.

In [18]:
tests = [
    "What is the time now?",
    "calculate 5 + 10 * 2",
    "Namaste",
    "time kya hai",
    "5 plus 7"
]

for t in tests:
    print(run_agent(t))

{'input': 'What is the time now?', 'intent': 'get_time', 'output': '2026-03-30 09:20:31 AM IST'}
{'input': 'calculate 5 + 10 * 2', 'intent': 'calculator', 'output': '25'}
{'input': 'Namaste', 'intent': 'greeting', 'output': 'Namaste! I am your agentic AI assistant.'}
{'input': 'time kya hai', 'intent': 'get_time', 'output': '2026-03-30 09:20:31 AM IST'}
{'input': '5 plus 7', 'intent': 'calculator', 'output': '12'}


## 5. Voice input with Whisper
Upload an audio file and transcribe it.

You can upload `.wav`, `.mp3`, `.m4a`, etc.

In [4]:
from google.colab import files
uploaded = files.upload()

audio_file = list(uploaded.keys())[0]
print("Uploaded file:", audio_file)

Saving Recording.m4a to Recording.m4a
Uploaded file: Recording.m4a


In [6]:
!pip install -q openai-whisper
!apt-get update -qq
!apt-get install -y ffmpeg -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.6 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [7]:
import whisper

model = whisper.load_model("base")
result = model.transcribe(audio_file)

transcribed_text = result["text"]
print("Transcribed text:", transcribed_text)

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 133MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed text:  Hello, who are you?


In [8]:
agent_result = run_agent(transcribed_text)
print(agent_result)

{'input': ' Hello, who are you?', 'intent': 'greeting', 'output': 'Namaste! I am your agentic AI assistant.'}


## 6. Optional: Generate a sample audio inside Colab
Use this if you do not want to upload your own audio.

In [9]:
!pip -q install gtts
from gtts import gTTS

sample_text = "What is the time now"
tts = gTTS(text=sample_text, lang="en")
tts.save("audio.wav")
print("Created audio.wav")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
Created audio.wav


In [10]:
result = model.transcribe("audio.wav")
print("Transcribed text:", result["text"])
print(run_agent(result["text"]))

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribed text:  What is the time now?
{'input': ' What is the time now?', 'intent': 'get_time', 'output': '2026-03-30 03:28:31'}


## 7. Results table for report/PPT


In [11]:
import pandas as pd

demo_inputs = [
    "What is the time now?",
    "calculate 12 + 8 / 2",
    "Namaste"
]

rows = []
for text in demo_inputs:
    out = run_agent(text)
    rows.append(out)

df = pd.DataFrame(rows)
df

,input,intent,output
0,What is the time now?,get_time,2026-03-30 03:28:50
1,calculate 12 + 8 / 2,calculator,16.0
2,Namaste,greeting,Namaste! I am your agentic AI assistant.



- **Problem:** Existing assistants have limited Indic-language automation.
- **Solution:** A voice-based agentic AI system using Whisper + LLM-style intent routing + tool execution.
- **Agentic behavior:** The system decides which tool to use based on user intent.
- **Current tools:** Time, calculator, greeting.
- **Future scope:** Maps, weather, reminders, app control, booking, multilingual support.

## 9. Our Workflow
1. Run the text demo first.
2. Show voice upload.
3. Ask:
   - "What is the time now?"
   - "Calculate 5 plus 10"
4. Explain that the agent:
   - understands the request,
   - selects a tool,
   - executes it,
   - returns the result.

##Thank you